## **Modeling**

### **Exploration**
- Goals
    + XGBoost regressor - count:poisson
    + LigthGBM - poisson
    + Sci-kit learn poisson
- Cards
    + Yellow card: LightGBM Regressor - tweedie, poisson || XGBoost regressor - count:poisson
    + Red card (0 & 1): Logistic Regression || XGBoost classifier
- Corners
    + LightGBM Regressor - regression
    + Sci-kit learn - Linear Regression || Ridge (L2 penalty)

### **Strategy**
1. Use train dataset to find suitable models for each task
2. Use GridSearchCV/RandomizedSearchCV to tune the selected models (combine train + val sets together)
    + Note: RandomizedSearchCV is more preferred. GridSearchCV for final refinement if necessary
3. Evaluate models with train set to check well-fit/overfit/underfit
4. Fix parameters in case models are overfit/underfit
5. Test models and re-evaluate with test set
6. Save models

### **Evaluation**

1. Goals
    + Poisson deviance (mean_poisson_deviance), MAE, RMSE
2. Yellow Cards
    + MAE, RMSE
3. Red Cards
    + Log los, ROC-AUC, F1-score, Accuracy
4. Corners
    + MAE, RMSE, Poission (if count model)

In [1]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_poisson_deviance, roc_auc_score, log_loss, f1_score, average_precision_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LogisticRegression, PoissonRegressor, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Evaluation function for goals
def eval_goal_model(name, task,model, X_train, y_train, X_test, y_test):

    # =========================
    # RESULTS REFRESH
    # =========================
    results = {}

    # =========================
    # PREDICTIONS
    # =========================
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # =========================
    # SAFETY CLIP
    # Poisson means must be > 0
    # =========================
    train_pred = np.clip(train_pred, 1e-6, None)
    test_pred = np.clip(test_pred, 1e-6, None)

    # =========================
    # METRICS
    # =========================
    if task == "goals":
        results = {
            "model": name,

            # TRAIN
            "train_mae": mean_absolute_error(y_train, train_pred),

            "train_rmse": np.sqrt(
                mean_squared_error(y_train, train_pred)
            ),

            "train_poisson": mean_poisson_deviance(
                y_train,
                train_pred
            ),

            # TEST (or VAL)
            "test_mae": mean_absolute_error(y_test, test_pred),

            "test_rmse": np.sqrt(
                mean_squared_error(y_test, test_pred)
            ),

            "test_poisson": mean_poisson_deviance(
                y_test,
                test_pred
            )
        }

    elif task == "yellow_cards":
        results = {
            "model": name,

            # TRAIN
            "train_mae": mean_absolute_error(y_train, train_pred),
            "train_rmse": np.sqrt(
                mean_squared_error(y_train, train_pred)
            ),

            # TEST
            "test_mae": mean_absolute_error(y_test, test_pred),
            "test_rmse": np.sqrt(
                mean_squared_error(y_test, test_pred)
            ),
        }

    elif task == "red_cards":
        # =========================
        # PROBABILITIES (IMPORTANT)
        # =========================
        train_prob = model.predict_proba(X_train)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        # =========================
        # BINARY CLASS (for F1/accuracy/recall)
        # =========================
        train_pred = (train_prob >= 0.5).astype(int)
        test_pred = (test_prob >= 0.5).astype(int)

        results = {
            "model": name,

            # TRAIN
            "train_clf_f1": f1_score(y_train, train_pred),
            
            "train_log_loss": log_loss(y_train, train_prob),
            "train_ap": average_precision_score(y_train, train_prob),
            "train_roc_auc": roc_auc_score(y_train, train_prob),

            # TEST
            "test_clf_f1": f1_score(y_test, test_pred),
            
            "test_log_loss": log_loss(y_test, test_prob),
            "test_ap": average_precision_score(y_test, test_prob),
            "test_roc_auc": roc_auc_score(y_test, test_prob),
        }

    elif task == "corners":
        results = {
            "model": name,
            "message": "if corners are not count models, please ignore the poisson metrics",

            # TRAIN
            "train_mae": mean_absolute_error(y_train, train_pred),
            "train_rmse": np.sqrt(
                mean_squared_error(y_train, train_pred)
            ),
            "train_poisson": mean_poisson_deviance(
                y_train,
                train_pred
            ),

            # TEST
            "test_mae": mean_absolute_error(y_test, test_pred),
            "test_rmse": np.sqrt(
                mean_squared_error(y_test, test_pred)
            ),

            "test_poisson": mean_poisson_deviance(
                y_test,
                test_pred
            )
        }

    return results

In [3]:
# Load data - use train_set and val_set only
mydf_train = pd.read_csv(r"data/train.csv")
mydf_val = pd.read_csv(r"data/val.csv")
mydf_test = pd.read_csv(r"data/test.csv")

In [4]:
# Splitting data
goal_corner_cols = ["home_elo", "away_elo", "elo_diff", "home_advantage", "home_attack_rate", "home_defense_rate", "away_attack_rate", "away_defense_rate"]
card_cols = goal_corner_cols + ["home_disc", "away_disc", "disc_diff", "disc_sum"]

X_goals_train = mydf_train[goal_corner_cols]
X_cards_train = mydf_train[card_cols]
y_home_goals_train = mydf_train["home_score"]
y_away_goals_train = mydf_train["away_score"]
y_home_yellow_train = mydf_train["home_yellow"]
y_away_yellow_train = mydf_train["away_yellow"]
y_home_red_train = mydf_train["home_red"]
y_away_red_train = mydf_train["away_red"]
y_home_corners_train = mydf_train["home_corners"]
y_away_corners_train = mydf_train["away_corners"]

X_goals_val = mydf_val[goal_corner_cols]
X_cards_val = mydf_val[card_cols]
y_home_goals_val = mydf_val["home_score"]
y_away_goals_val = mydf_val["away_score"]
y_home_yellow_val = mydf_val["home_yellow"]
y_away_yellow_val = mydf_val["away_yellow"]
y_home_red_val = mydf_val["home_red"]
y_away_red_val = mydf_val["away_red"]
y_home_corners_val = mydf_val["home_corners"]
y_away_corners_val = mydf_val["away_corners"]

X_goals_test = mydf_test[goal_corner_cols]
X_cards_test = mydf_test[card_cols]
y_home_goals_test = mydf_test["home_score"]
y_away_goals_test = mydf_test["away_score"]
y_home_yellow_test = mydf_test["home_yellow"]
y_away_yellow_test = mydf_test["away_yellow"]
y_home_red_test = mydf_test["home_red"]
y_away_red_test = mydf_test["away_red"]
y_home_corners_test = mydf_test["home_corners"]
y_away_corners_test = mydf_test["away_corners"]

#### **Goal models**

In [5]:
# Find a suitable model
sk_home = PoissonRegressor()
sk_away = PoissonRegressor()

xgb_home = XGBRegressor(objective="count:poisson")
xgb_away = XGBRegressor(objective="count:poisson")

lgb_home = LGBMRegressor(objective="poisson", force_row_wise=True)
lgb_away = LGBMRegressor(objective="poisson", force_row_wise=True)

In [6]:
# Fit and evaluate models
models = {
    "sk_home": sk_home,
    "sk_away": sk_away,
    "xgb_home": xgb_home,
    "xgb_away": xgb_away,
    "lgb_home": lgb_home,
    "lgb_away": lgb_away
}

results_list = []

for name, model in models.items():

    # =========================
    # HOME MODELS
    # =========================
    if "home" in name:

        y_train = y_home_goals_train
        y_test = y_home_goals_val

    # =========================
    # AWAY MODELS
    # =========================
    else:

        y_train = y_away_goals_train
        y_test = y_away_goals_val

    # =========================
    # FIT MODEL
    # =========================
    model.fit(X_goals_train, y_train)

    # =========================
    # EVALUATE
    # =========================
    results = eval_goal_model(
        name,
        "goals",
        model,
        X_goals_train,
        y_train,
        X_goals_val,
        y_test
    )

    results_list.append(results)

# =========================
# RESULTS TABLE
# =========================
results_df = pd.DataFrame(results_list)

results_df = results_df.sort_values(
    "test_poisson",
    ascending=True
)

results_df


[LightGBM] [Info] Total Bins 1668
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 8
[LightGBM] [Info] Start training from score 0.591178
[LightGBM] [Info] Total Bins 1668
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 8
[LightGBM] [Info] Start training from score 0.143208


,model,train_mae,train_rmse,train_poisson,test_mae,test_rmse,test_poisson
5,lgb_away,0.839920,1.127611,1.124556,0.835476,1.120154,1.112443
3,xgb_away,0.748689,0.988808,0.950158,0.837538,1.126338,1.129126
4,lgb_home,1.057336,1.426113,1.172711,1.019123,1.324026,1.171555
2,xgb_home,0.936599,1.238914,0.973843,1.029455,1.346744,1.190346
1,sk_away,0.993568,1.418538,1.583310,0.976095,1.342050,1.510376
0,sk_home,1.332142,1.846266,1.775518,1.272601,1.642373,1.687411


######################### GOAL MODEL CONCLUSION #############################

The models for goal prediction, we should prioritize the poission metric

=> LBG with the poisson objective for both home and away is the best for goal prediction

#### **Yellow card models**

In [7]:
# Find a suitable model
lgb_poisson_home = LGBMRegressor(objective="poisson", force_row_wise=True)
lgb_poisson_away = LGBMRegressor(objective="poisson", force_row_wise=True)

lgb_tweedie_home = LGBMRegressor(objective="tweedie", force_row_wise=True)
lgb_tweedie_away = LGBMRegressor(objective="tweedie", force_row_wise=True)

xgb_poisson_home = XGBRegressor(objective="count:poisson")
xgb_poisson_away = XGBRegressor(objective="count:poisson")

In [8]:
models = {
    "lgb_poisson_home": lgb_poisson_home,
    "lgb_poisson_away": lgb_poisson_away,
    "lgb_tweedie_home": lgb_tweedie_home,
    "lgb_tweedie_away": lgb_tweedie_away,
    "xgb_poisson_home": xgb_poisson_home,
    "xgb_poisson_away": xgb_poisson_away,
}

results_list = []

for name, model in models.items():

    # =========================
    # HOME MODELS
    # =========================
    if "home" in name:

        y_train = y_home_yellow_train
        y_test = y_home_yellow_val

    # =========================
    # AWAY MODELS
    # =========================
    else:

        y_train = y_away_yellow_train
        y_test = y_away_yellow_val

    # =========================
    # FIT MODEL
    # =========================
    model.fit(X_cards_train, y_train)

    # =========================
    # EVALUATE
    # =========================
    results = eval_goal_model(
        name,
        "yellow_cards",
        model,
        X_cards_train,
        y_train,
        X_cards_val,
        y_test
    )

    results_list.append(results)

# =========================
# RESULTS TABLE
# =========================
results_df = pd.DataFrame(results_list)

results_df = results_df.sort_values(
    "test_mae",
    ascending=True
)

results_df

[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] Start training from score 0.811148
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] Start training from score 0.739472
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] Start training from score 0.811148
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] Start training from score 0.739472


,model,train_mae,train_rmse,test_mae,test_rmse
1,lgb_poisson_away,1.096618,1.395650,1.114376,1.432544
3,lgb_tweedie_away,1.065908,1.355931,1.122682,1.438566
5,xgb_poisson_away,1.010429,1.279837,1.137780,1.450425
0,lgb_poisson_home,1.162458,1.458828,1.194399,1.504577
2,lgb_tweedie_home,1.124369,1.418565,1.195049,1.510029
4,xgb_poisson_home,1.075785,1.352985,1.203589,1.517366


######################### YELLOW CARD MODEL CONCLUSION #############################

The models for yellow card prediction, we should prioritize the mae metric

=> LBG with the poisson objective for both home and away is the best for yellow card prediction

#### **Red card models**

In [9]:
# Find a suitable model
lgb_logistic_home = LGBMClassifier(objective="binary", force_row_wise=True)
lgb_logistic_away = LGBMClassifier(objective="binary", force_row_wise=True)

sk_logistic_home = LogisticRegression()
sk_logistic_away = LogisticRegression()

In [10]:
models = {
    "lgb_logistic_home": lgb_logistic_home,
    "lgb_logistic_away": lgb_logistic_away,
    "sk_logistic_home": sk_logistic_home,
    "sk_logistic_away": sk_logistic_away
}

results_list = []

for name, model in models.items():

    # =========================
    # HOME MODELS
    # =========================
    if "home" in name:

        y_train = y_home_red_train
        y_test = y_home_red_val

    # =========================
    # AWAY MODELS
    # =========================
    else:

        y_train = y_away_red_train
        y_test = y_away_red_val

    # =========================
    # FIT MODEL
    # =========================
    model.fit(X_cards_train, y_train)

    # =========================
    # EVALUATE
    # =========================
    results = eval_goal_model(
        name,
        "red_cards",
        model,
        X_cards_train,
        y_train,
        X_cards_val,
        y_test
    )

    results_list.append(results)

# =========================
# RESULTS TABLE
# =========================
results_df = pd.DataFrame(results_list)

results_df = results_df.sort_values(
    "test_ap",
    ascending=True,
)

results_df

[LightGBM] [Info] Number of positive: 1618, number of negative: 20854
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.072001 -> initscore=-2.556355
[LightGBM] [Info] Start training from score -2.556355
[LightGBM] [Info] Number of positive: 1644, number of negative: 20828
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.073158 -> initscore=-2.539166
[LightGBM] [Info] Start training from score -2.539166


,model,train_clf_f1,train_log_loss,train_ap,train_roc_auc,test_clf_f1,test_log_loss,test_ap,test_roc_auc
3,sk_logistic_away,0.000000,0.263936,0.074558,0.503859,0.0,0.259130,0.065734,0.476441
0,lgb_logistic_home,0.058788,0.195809,0.672475,0.926652,0.0,0.270541,0.071976,0.493389
1,lgb_logistic_away,0.046346,0.195807,0.688075,0.930681,0.0,0.260191,0.077702,0.526537
2,sk_logistic_home,0.000000,0.262101,0.069929,0.483487,0.0,0.263485,0.078330,0.502804


######################### RED CARD MODEL CONCLUSION #############################

The models for red card prediction, we should prioritize the average_precision metric

=> Sklearn Logistic Regression for both home and away is the best for red card prediction

#### **Corners models**

In [11]:
# Find a suitable model
lgb_regressor_home = LGBMRegressor(objective="regression")
lgb_regressor_away = LGBMRegressor(objective="regression")

sk_linear_home = LinearRegression()
sk_linear_away = LinearRegression()

sk_ridge_home = Ridge()
sk_ridge_away = Ridge()

In [12]:
models = {
    "lgb_regressor_home": lgb_regressor_home,
    "lgb_regressor_away": lgb_regressor_away,
    "sk_linear_home": sk_linear_home,
    "sk_linear_away": sk_linear_away,
    "sk_ridge_home": sk_ridge_home,
    "sk_ridge_away": sk_ridge_away
}

results_list = []

for name, model in models.items():

    # =========================
    # HOME MODELS
    # =========================
    if "home" in name:

        y_train = y_home_corners_train
        y_test = y_home_corners_val

    # =========================
    # AWAY MODELS
    # ========================= 
    else:

        y_train = y_away_corners_train
        y_test = y_away_corners_val

    # =========================
    # FIT MODEL
    # =========================
    model.fit(X_cards_train, y_train)

    # =========================
    # EVALUATE
    # =========================
    results = eval_goal_model(
        name,
        "corners",
        model,
        X_cards_train,
        y_train,
        X_cards_val,
        y_test
    )

    results_list.append(results)

# =========================
# RESULTS TABLE
# =========================
results_df = pd.DataFrame(results_list)

results_df = results_df.sort_values(
    "test_mae",
    ascending=True
)

results_df

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001989 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] Start training from score 5.046057
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000376 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 22472, number of used features: 12
[LightGBM] [Info] Start training from score 4.678622


,model,message,train_mae,train_rmse,train_poisson,test_mae,test_rmse,test_poisson
3,sk_linear_away,"if corners are not count models, please ignore...",1.717190,2.169850,1.085321,1.724234,2.189094,1.063991
5,sk_ridge_away,"if corners are not count models, please ignore...",1.717722,2.169860,1.084480,1.724667,2.188935,1.063831
1,lgb_regressor_away,"if corners are not count models, please ignore...",1.603359,2.023716,0.938237,1.745477,2.212045,1.084559
2,sk_linear_home,"if corners are not count models, please ignore...",1.796032,2.256231,1.092846,1.805759,2.262898,1.068503
4,sk_ridge_home,"if corners are not count models, please ignore...",1.796121,2.256233,1.092850,1.805827,2.262911,1.068518
0,lgb_regressor_home,"if corners are not count models, please ignore...",1.680879,2.110622,0.943042,1.820678,2.283498,1.088417


######################### CORNER MODEL CONCLUSION #############################

The models for corner prediction, we should prioritize the mae metric

=> Sklearn Linear Regression for both home and away is the best for corner prediction

We can replace Linear Regression with Ridge (L2 regularization/penalty)

### **Build final models and fine tuning**

In [13]:
# =========================
# GOAL MODELS
# =========================
goal_home = LGBMRegressor(objective="poisson", force_row_wise=True)
goal_away = LGBMRegressor(objective="poisson", force_row_wise=True)


# =========================
# YELLOW CARD MODELS
# =========================
yellow_home = LGBMRegressor(objective="poisson", force_row_wise=True)
yellow_away = LGBMRegressor(objective="poisson", force_row_wise=True)


# =========================
# RED CARD MODELS
# =========================
red_home = LogisticRegression()
red_away = LogisticRegression()


# =========================
# CORNER MODELS
# =========================
corner_home = Ridge()
corner_away = Ridge()


tscv = TimeSeriesSplit(n_splits=5)

In [14]:
param_grid_poisson = {
    "n_estimators": [100, 300, 500, 800],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 5, 10, 15],
    "min_child_samples": [10, 20, 50, 100],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 0.5, 1.0],
    "reg_lambda": [0, 0.1, 0.5, 1.0]
}

param_grid_log = [
    {
        "penalty": ["l1"],
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear", "saga"],
        "class_weight": [None, "balanced"],
        "max_iter": [300, 500]
    },
    {
        "penalty": ["l2"],
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs", "liblinear", "saga"],
        "class_weight": [None, "balanced"],
        "max_iter": [300, 500]
    },
    {
        "penalty": ["elasticnet"],
        "C": [0.01, 0.1, 1, 10],
        "solver": ["saga"],
        "l1_ratio": [0.2, 0.5, 0.8],
        "class_weight": [None, "balanced"],
        "max_iter": [500, 1000]
    }
]

param_grid_ridge = {
    "alpha": [0.1, 1, 5, 10, 50, 100],
    "fit_intercept": [True, False],
    "solver": ["auto", "svd", "cholesky", "lsqr"]
}

In [15]:
goal_home_search = RandomizedSearchCV(
    goal_home,
    param_distributions=param_grid_poisson,
    n_iter=40,
    scoring="neg_mean_poisson_deviance",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)


goal_away_search = RandomizedSearchCV(
    goal_away,
    param_distributions=param_grid_poisson,
    n_iter=40,
    scoring="neg_mean_poisson_deviance",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

In [16]:
yellow_home_search = RandomizedSearchCV(
    yellow_home,
    param_distributions=param_grid_poisson,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)


yellow_away_search = RandomizedSearchCV(
    yellow_away,
    param_distributions=param_grid_poisson,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

In [17]:
red_home_search = RandomizedSearchCV(
    red_home,
    param_distributions=param_grid_log,
    n_iter=40,
    scoring="average_precision",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)


red_away_search = RandomizedSearchCV(
    red_away,
    param_distributions=param_grid_log,
    n_iter=40,
    scoring="average_precision",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

In [18]:
corner_home_search = RandomizedSearchCV(
    corner_home,
    param_distributions=param_grid_ridge,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)


corner_away_search = RandomizedSearchCV(
    corner_away,
    param_distributions=param_grid_ridge,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

In [19]:
X_goals_train_full = pd.concat(
    [X_goals_train, X_goals_val],
    axis=0
)

X_goals_train_full = X_goals_train_full.loc[:, goal_corner_cols]


y_home_goals_train_full = pd.concat(
    [y_home_goals_train, y_home_goals_val],
    axis=0
)

y_away_goals_train_full = pd.concat(
    [y_away_goals_train, y_away_goals_val],
    axis=0
)

In [20]:
X_cards_train_full = pd.concat(
    [X_cards_train, X_cards_val],
    axis=0
)

X_cards_train_full = X_cards_train_full.loc[:, card_cols]


# YELLOW
y_home_yellow_train_full = pd.concat(
    [y_home_yellow_train, y_home_yellow_val],
    axis=0
)

y_away_yellow_train_full = pd.concat(
    [y_away_yellow_train, y_away_yellow_val],
    axis=0
)

# RED
y_home_red_train_full = pd.concat(
    [y_home_red_train, y_home_red_val],
    axis=0
)

y_away_red_train_full = pd.concat(
    [y_away_red_train, y_away_red_val],
    axis=0
)


# CORNER
y_home_corners_train_full = pd.concat(
    [y_home_corners_train, y_home_corners_val],
    axis=0
)

y_away_corners_train_full = pd.concat(
    [y_away_corners_train, y_away_corners_val],
    axis=0
)

In [21]:
# =========================
# GOALS
# =========================
goal_home_best = goal_home_search.fit(
    X_goals_train_full, y_home_goals_train_full
).best_estimator_

goal_away_best = goal_away_search.fit(
    X_goals_train_full, y_away_goals_train_full
).best_estimator_


# =========================
# YELLOW CARDS
# =========================
yellow_home_best = yellow_home_search.fit(
    X_cards_train_full, y_home_yellow_train_full
).best_estimator_

yellow_away_best = yellow_away_search.fit(
    X_cards_train_full, y_away_yellow_train_full
).best_estimator_


# =========================
# RED CARDS
# =========================
red_home_best = red_home_search.fit(
    X_cards_train_full, y_home_red_train_full
).best_estimator_

red_away_best = red_away_search.fit(
    X_cards_train_full, y_away_red_train_full
).best_estimator_


# =========================
# CORNERS
# =========================
corner_home_best = corner_home_search.fit(
    X_goals_train_full, y_home_corners_train_full
).best_estimator_

corner_away_best = corner_away_search.fit(
    X_goals_train_full, y_away_corners_train_full
).best_estimator_

Fitting 5 folds for each of 40 candidates, totalling 200 fits
[LightGBM] [Info] Total Bins 1668
[LightGBM] [Info] Number of data points in the train set: 26515, number of used features: 8
[LightGBM] [Info] Start training from score 0.571905
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

In [22]:
results_list = []

models = {
    "goal_home": goal_home_best,
    "goal_away": goal_away_best,
    "yellow_home": yellow_home_best,
    "yellow_away": yellow_away_best,
    "red_home": red_home_best,
    "red_away": red_away_best,
    "corner_home": corner_home_best,
    "corner_away": corner_away_best
}


for name, model in models.items():

    if "goal" in name:
        task = "goals"
        X_train, y_train = X_goals_train_full, y_home_goals_train_full if "home" in name else y_away_goals_train_full
        X_test, y_test = X_goals_test, y_home_goals_test if "home" in name else y_away_goals_test

    elif "yellow" in name:
        task = "yellow_cards"
        X_train, y_train = X_cards_train_full, y_home_yellow_train_full if "home" in name else y_away_yellow_train_full
        X_test, y_test = X_cards_test, y_home_yellow_test if "home" in name else y_away_yellow_test

    elif "red" in name:
        task = "red_cards"
        X_train, y_train = X_cards_train_full, y_home_red_train_full if "home" in name else y_away_red_train_full
        X_test, y_test = X_cards_test, y_home_red_test if "home" in name else y_away_red_test

    else:
        task = "corners"
        X_train, y_train = X_goals_train_full, y_home_corners_train_full if "home" in name else y_away_corners_train_full
        X_test, y_test = X_goals_test, y_home_corners_test if "home" in name else y_away_corners_test

    results = eval_goal_model(name, task, model, X_train, y_train, X_test, y_test)
    results_list.append(results)


# =========================
# RESULTS TABLE
# =========================
results_df = pd.DataFrame(results_list)

results_df.to_csv("data/best_models_results.csv")

In [23]:
# Save model
import pickle


# # Load the model back later
# with open('model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)


models = {
    "goal_home": goal_home_best,
    "goal_away": goal_away_best,
    "yellow_home": yellow_home_best,
    "yellow_away": yellow_away_best,
    "red_home": red_home_best,
    "red_away": red_away_best,
    "corner_home": corner_home_best,
    "corner_away": corner_away_best
}

for name, model in models.items():
    with open(f'models/{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
